# Acoustic Scan 

In [10]:
# matching pursuit
# depth profiling
# attenuation with high f. reflection ok, transmission no
# look at acoustic resonances, dip in attenuation
# 

In [11]:
%load_ext autoreload
%autoreload 2
import numpy as np
from matplotlib import pyplot as plt
import sys


sys.path.append('..') # path to the src directory
sys.path.append('/Users/xz498/Desktop/ultrasound project/data analysis/ultrasonicTesting')
sys.path.append('/Users/xz498/Desktop/ultrasound project/data analysis/M3Learning-Util/src')
sys.path.append('/Users/xz498/Desktop/ultrasound project/data analysis/AutoPhysLearn/src')
sys.path.append('/Users/xz498/Desktop/ultrasound project/data analysis/ultrasonic_ml/src')


from scipy.signal import butter, sosfiltfilt
import copy
import math
import time
from tqdm import tqdm
import pickleJar as pj
import tomography as tm

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from viz.visualize_scan_data import *
from IPython.display import display
import plotly.graph_objects as go

## Dataloader with preprocessing

In [13]:
from data import datasets
from data.datasets import morlet_1D_dataset_real

dset = morlet_1D_dataset_real(sq3lite_path='/Users/xz498/Desktop/ultrasound project/data analysis/example data/SA_tomography_foil_tape_scan.sqlite3',
                              dset_name='voltage_transmission_forward',
                              image_shape = (1,1),
                            #   crops = [(0,4000)] #(15000,19000)
                            ) 

sqliteToPickle Warning: pickle file /Users/xz498/Desktop/ultrasound project/data analysis/example data/SA_tomography_foil_tape_scan.pickle already exists. Conversion aborted.


/Users/xz498/Desktop/ultrasound project/data analysis/ultrasonicTesting/pickleJar.py:1185: RuntimeWarning: divide by zero encountered in log10
  logData = np.log10(abs(data))


In [14]:
# dset.display_dict_tree()

## Interactive Viewer with Slider

Use the slider below to browse through all scans interactively.

In [15]:
# Create interactive viewer with slider (fast - uses ipywidgets)
from viz.visualize_scan_data import plotly_viewer
viewer = plotly_viewer(dset)
display(viewer)  # or just: viewer  (in Jupyter, the last line auto-displays)

    'data': [{'mode': 'lines…

## try training model with morlet packet

goals:
- figure out mean position and f of morlet packet
- using this, calculate speed of sound in this water

In [16]:
import torch
x = torch.randn(10,10)
x.device

device(type='cpu')

In [17]:
from models.morlet_fitter import Fitter_AE, morlet_1D_fitters_real
from autophyslearn.spectroscopic.nn import block_factory, Conv_Block, FC_Block  # pyright: ignore[reportMissingImports]
from autophyslearn.spectroscopic.nn import Multiscale1DFitter
# from data.custom_sampler import Gaussian_Sampler
import torch

num_fits = 2 # number of curves to sum up
num_params = 4 # number of parameters to fit
# todo: change wandb naming to include noise level, group and regularization technique
# todo: test more num fits
model = Fitter_AE(function=morlet_1D_fitters_real,
                dset=dset,
                num_params=num_params,
                num_fits=num_fits,
                checkpoints_label='ultrasound_water',
                input_channels = 1,
                learning_rate=3e-6,
                device='cpu',
                encoder = Multiscale1DFitter,
                encoder_params = {
                    "model_block_dict": { # factory wrapper for blocks
                            "hidden_x1": block_factory(Conv_Block)(output_channels_list=[128,128], 
                                                                    kernel_size_list=[5,5], 
                                                                    pool_list=[2000,500], 
                                                                    max_pool=False),
                            # "hidden_xfc": block_factory(FC_Block)(output_size_list=[128,64]), # remove 2nd block and skip connections
                            # "hidden_x2": block_factory(Conv_Block)(output_channels_list=[32,16], 
                            #                                         kernel_size_list=[75,75], 
                            #                                         pool_list=[64,32], 
                            #                                         max_pool=True),
                            "hidden_embedding": block_factory(FC_Block)(output_size_list=[8*num_fits,num_params*num_fits], last=True),
                        },
                        # TEST: LIMITS,
                        # "skip_connections": {'hidden_xfc': 'hidden_embedding'},
                        "skip_connections": {},
                        "function_kwargs": {'limits': [1, # amplitude
                                                       dset.spec_len, # mean
                                                       dset.spec_len/10, # stdev
                                                       1e-2] # freq max (100 MHz)
                                            } 
                    },
                )


### Train model for several epochs


In [21]:
# import wandb
# wandb.init(group='sub_sampler_type', name='sub_noise_level') # later change config for regularization

model.train(epochs=1,save_every=1, batch_size=1, log_wandb=False, 
            # lr_scheduling=True,
            coef1=1e-3
            )

/Users/xz498/Desktop/ultrasound project/data analysis/example data/ultrasound_water/checkpoints/voltage_transmission_forward


100%|██████████| 1911/1911 [00:21<00:00, 88.92it/s]


Epoch: 000/001 | Train Loss: 0.0012
.............................


### Embeddings

In [22]:
def write_scaled_embedding(batch_size=1):
    for i, (idx, x) in enumerate(tqdm(model.dataloader, leave=True, total=len(model.dataloader))):
        with torch.no_grad():
            fits, params = model.encoder(x.float().to(model.device))
            fits = fits.cpu().numpy()
            params = params.cpu().numpy()
    return fits, params

fits, params = write_scaled_embedding(batch_size=1)

# sweep frequencies and search for resonances 
# attenuation in each layer accounts for spherical nature of wave
# data for one to 20 layers
# measure waveform from 30-50, measuring thermal gradient. shoul dbe the same as 40 (mean), so why isnt it?
# goal: use ultrasound to monitor the thermal expansion so we can decreasing charging rate. this way the battery is less likely to experience stress and can cycle more

100%|██████████| 1911/1911 [00:07<00:00, 272.98it/s]


In [23]:
from viz.visualize_scan_data import training_viewer

training_viewer(dset, fits, params, idx=0)
    

In [ ]:
debug

    [... skipping 1 hidden frame(s)]
  /var/folders/vg/g8h80y317zj5sfz8_f4l85wddxx210/T/ipykernel_71805/3883092592.py(3)<module>()
      1 from viz.visualize_scan_data import training_viewer
      2 
----> 3 training_viewer(dset, fits, params, idx=1)
      4 
> /Users/xz498/Desktop/ultrasound project/data analysis/ultrasonic_ml/src/viz/visualize_scan_data.py(29)training_viewer()
     27 
     28     return data, numeric_keys
---> 29 
     30 try:
     31     import ipywidgets as widgets

